# TP7 - Exercise 2 : Poisson Solver — Speedup & Efficiency Analysis

This notebook:
1. Compiles `poisson_solver.c`
2. Runs it with increasing process counts
3. Plots **speedup** S(p) = T(1)/T(p) and **efficiency** E(p) = S(p)/p

> **Before running:** make sure `poisson_solver.c` was compiled with `make` in the same directory.

In [ ]:
import subprocess
import re
import matplotlib.pyplot as plt
import numpy as np

# ---- Configuration ----
GRID_SIZE      = 500          # Use a large grid to see parallelism benefits
PROCESS_COUNTS = [1, 2, 4, 8] # Adjust to the number of cores on your VM
BINARY         = './poisson'

# ---- Compile first ----
print('Compiling poisson_solver.c ...')
ret = subprocess.run('make', shell=True, capture_output=True, text=True)
print(ret.stdout)
if ret.returncode != 0:
    print('Compilation FAILED:', ret.stderr)
else:
    print('Compilation OK')

In [ ]:
times        = []
iterations   = []

print(f'Grid size: {GRID_SIZE} x {GRID_SIZE}\n')

for p in PROCESS_COUNTS:
    cmd = f'mpirun --oversubscribe -n {p} {BINARY} {GRID_SIZE} {GRID_SIZE}'
    print(f'  Running with {p} process(es) ...')
    try:
        out = subprocess.check_output(cmd, shell=True,
                                      stderr=subprocess.STDOUT,
                                      universal_newlines=True)
        m_time = re.search(r'Converged after (\d+) iterations in ([\d.]+) seconds', out)
        if m_time:
            iters = int(m_time.group(1))
            t     = float(m_time.group(2))
            times.append(t)
            iterations.append(iters)
            print(f'    -> {iters} iterations, {t:.4f} s')
        else:
            print('    -> Could not parse output')
            times.append(None)
            iterations.append(None)
    except subprocess.CalledProcessError as e:
        print(f'    -> ERROR: {e.output}')
        times.append(None)
        iterations.append(None)

In [ ]:
# ---- Compute speedup and efficiency ----
T1 = times[0]  # serial (1-process) time

valid_p  = []
speedups = []
effics   = []

for p, t in zip(PROCESS_COUNTS, times):
    if t is not None and T1 is not None:
        S = T1 / t
        E = S / p
        valid_p.append(p)
        speedups.append(S)
        effics.append(E)
        print(f'  p={p:2d}  T={t:.4f}s  S={S:.2f}  E={E:.2f}')

# ---- Plot ----
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f'Poisson Solver — Grid {GRID_SIZE}×{GRID_SIZE}', fontsize=14)

# Speedup
ax = axes[0]
ax.plot(valid_p, speedups, 'o-', color='steelblue', linewidth=2, label='Measured speedup')
ax.plot(valid_p, valid_p,  '--', color='tomato',    linewidth=1.5, label='Ideal (linear)')
ax.set_xlabel('Number of processes  p')
ax.set_ylabel('Speedup  S(p) = T(1)/T(p)')
ax.set_title('Speedup')
ax.legend()
ax.grid(True, alpha=0.4)
ax.set_xticks(valid_p)

# Efficiency
ax = axes[1]
ax.plot(valid_p, effics, 'o-', color='seagreen', linewidth=2, label='Measured efficiency')
ax.axhline(y=1.0, color='tomato', linestyle='--', linewidth=1.5, label='Ideal (E=1)')
ax.set_xlabel('Number of processes  p')
ax.set_ylabel('Efficiency  E(p) = S(p)/p')
ax.set_title('Efficiency')
ax.set_ylim(0, 1.2)
ax.legend()
ax.grid(True, alpha=0.4)
ax.set_xticks(valid_p)

plt.tight_layout()
plt.savefig('poisson_speedup_analysis.png', dpi=150)
plt.show()
print('Saved: poisson_speedup_analysis.png')